# Owen Coder 7B - Kaggle Trainer (2x T4)

Runs **alongside** the Colab 3B notebook. Colab trains the 3B on 1x T4;
this trains the 7B on Kaggle's **2x T4 (30GB)** at the same time.

## One-time Kaggle setup (do this yourself)
1. Sign in at kaggle.com, then **Settings -> Phone verify** (unlocks GPU + internet).
2. New Notebook -> **Settings (right panel)**:
   - Accelerator: **GPU T4 x2**
   - Internet: **On**
3. **Add Data -> Upload** `training_data_merged.jsonl` as a private Dataset
   (it will mount under `/kaggle/input/<your-dataset-slug>/`).
4. Run all cells. When done, **Save Version** to download the GGUF from Output.


## 1. Environment check - confirm 2x T4


In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
print('GPU count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  [{i}] {p.name}  {p.total_memory/1e9:.1f} GB')
# Expect: GPU count: 2, two Tesla T4 @ ~15.8 GB each


## 2. Install Unsloth


In [ ]:
!pip install -q -U 'unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git' datasets trl 2>/dev/null || \
  pip install -q -U unsloth datasets trl
print('installed')


## 3. Load training data
Searches `/kaggle/input` (uploaded dataset) first, then `/kaggle/working`.


In [ ]:
import json, os, glob

CANDIDATES = []
CANDIDATES += glob.glob('/kaggle/input/**/training_data_merged.jsonl', recursive=True)
CANDIDATES += glob.glob('/kaggle/input/**/training_data.jsonl', recursive=True)
CANDIDATES += ['training_data_merged.jsonl', 'training_data.jsonl']
TRAINING_DATA = next((p for p in CANDIDATES if os.path.exists(p)), None)
assert TRAINING_DATA, 'Upload training_data_merged.jsonl as a Kaggle Dataset (Add Data -> Upload).'

rows = []
with open(TRAINING_DATA, encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            rows.append(json.loads(line))
print(f'Loaded {len(rows)} examples from {TRAINING_DATA}')


## 4. 7B config - tuned for 2x T4 headroom
Kaggle's 30GB lets us push longer sequences and rank than the Colab 3B run.


In [ ]:
BASE_MODELS = [
    'huihui_ai/Qwen2.5-Coder-7B-Instruct-abliterated',
    'Qwen/Qwen2.5-Coder-7B-Instruct',
]
MAX_SEQ_LENGTH = 4096   # 2x the Colab 7B run - Kaggle has the VRAM
LORA_R = 64
LORA_ALPHA = 128
LORA_DROPOUT = 0.05
EPOCHS = 4
BATCH_SIZE = 2          # 2x T4 headroom allows batch 2
GRAD_ACCUM = 8          # effective batch = 16
LR = 5e-5
OUT_DIR = '/kaggle/working'


## 5. Load 7B in 4-bit
Unsloth loads on GPU 0; the second T4 gives memory headroom for the merge/export
step below. (Set `CUDA_VISIBLE_DEVICES` before this cell only if you want to pin.)


In [ ]:
from unsloth import FastLanguageModel
import torch, gc
gc.collect(); torch.cuda.empty_cache()

model = tokenizer = None
for cand in BASE_MODELS:
    try:
        print('Trying', cand, '...')
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=cand, max_seq_length=MAX_SEQ_LENGTH,
            dtype=None, load_in_4bit=True)
        LOADED = cand; print('Loaded', cand); break
    except Exception as e:
        print('  failed:', str(e)[:200])
assert model is not None, 'Could not load any base model - check Internet is On.'

model = FastLanguageModel.get_peft_model(
    model, r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=['q_proj','k_proj','v_proj','o_proj',
                    'gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing='unsloth', random_state=42)
print('LoRA attached')


## 6. Format rows (Qwen chat template) -> dataset


In [ ]:
SYSTEM = ('You are Owen Coder, a security-focused code auditor. You find real,
          exploitable vulnerabilities and ignore noise.')

def format_row(r):
    instr = r.get('instruction', '')
    out = r.get('output', '')
    msgs = [{'role':'system','content':SYSTEM.replace(chr(10),' ')},
            {'role':'user','content':instr},
            {'role':'assistant','content':out}]
    return tokenizer.apply_chat_template(msgs, tokenize=False)

from datasets import Dataset
dataset = Dataset.from_list([{'text': format_row(r)} for r in rows])
print('Dataset:', len(dataset), 'examples')
print('Sample tokens:', tokenizer(dataset[0]['text'], return_tensors='pt')['input_ids'].shape[1])


## 7. Train


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=dataset,
    dataset_text_field='text', max_seq_length=MAX_SEQ_LENGTH,
    args=TrainingArguments(
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_ratio=0.05, num_train_epochs=EPOCHS,
        learning_rate=LR, fp16=True, logging_steps=10,
        optim='adamw_8bit', weight_decay=0.01,
        lr_scheduler_type='cosine', seed=42,
        output_dir=os.path.join(OUT_DIR,'ckpt'), report_to='none'))

stats = trainer.train()
print(stats)


## 8. Save adapter + export GGUF (q4_k_m) to /kaggle/working
After the run, **Save Version** and grab the `.gguf` from the Output tab.


In [ ]:
import gc, torch
gc.collect(); torch.cuda.empty_cache()

model.save_pretrained(os.path.join(OUT_DIR,'owen-coder-7b-lora'))
tokenizer.save_pretrained(os.path.join(OUT_DIR,'owen-coder-7b-lora'))
print('LoRA adapter saved')

print('Exporting merged GGUF (q4_k_m) - ~5-10 min ...')
model.save_pretrained_gguf(
    os.path.join(OUT_DIR,'owen-coder-7b-gguf'),
    tokenizer, quantization_method='q4_k_m')
print('Done. Download owen-coder-7b-gguf/*.gguf from the Output tab.')


## 9. Local install (after download)
```bash
# Put the .gguf next to Modelfile.7b, then:
ollama create owen-coder-7b -f Modelfile.7b
ollama run owen-coder-7b
```
